
# 🧠 Lab 5 – RAG Build + Chunking Experiment

In this lab, you’ll **build, test, and evaluate** a small **Retrieval-Augmented Generation (RAG)** pipeline.  
You’ll explore how **chunk size** and **top-k retrieval** affect system performance, measured by **recall@k** and **hallucination rate**.

---

By the end of this lab, you’ll be able to:
1. Describe how a RAG system retrieves and generates answers.
2. Implement and test a simple RAG pipeline.
3. Experiment with chunk size and number of retrieved documents (k).
4. Measure recall@k and hallucination rate.
5. Reflect on tradeoffs between retrieval and generation quality.


In [ ]:

# @title 🔧 Lab 5 Setup (run this first)

import sys
from pathlib import Path

COURSE_REPO_URL = "https://github.com/tulane-intro-ai-engineering/main.git"
COURSE_DIR = Path("/content/main")

# Clone or update the course repo inside this Colab runtime
if "google.colab" in sys.modules:
    if COURSE_DIR.exists():
        %cd {COURSE_DIR}
        !git pull --ff-only > /dev/null
    else:
        %cd /content
        !git clone --depth=1 {COURSE_REPO_URL} main
        %cd main
else:
    if not COURSE_DIR.exists():
        COURSE_DIR = Path(".").resolve()

if str(COURSE_DIR) not in sys.path:
    sys.path.append(str(COURSE_DIR))

from course_utils import lab5_colab_bootstrap

lab5_colab_bootstrap()

print("✅ Lab 5 setup complete.")
print("📁 Repo root:", COURSE_DIR)



## Pre-Lab Questions

Answer briefly before coding:

1. What does the “R” in RAG stand for, and what is its role?
2. What do you think happens if chunk size is too small or too large?
3. What is one way we might detect hallucination in generated answers?

> Edit this cell and type your answers below each question.



## Scientific Question & Hypothesis

**Question:**  
If we change chunk size and top-k, how do recall@k and hallucination rate change?

**Hypothesis:**  
I expect that as chunk size increases, recall@k may (increase/decrease) because …  
I also expect hallucination rate to … because …



## Part 1 – Build the RAG Pipeline

A Retrieval-Augmented Generation (RAG) system combines two main stages:

1. **Retrieve** relevant context (e.g., text chunks) based on a query.  
2. **Generate** an answer using both the retrieved context and the query.

We'll use helper functions from `course_utils`, but you'll fill in key steps.


In [ ]:

# Step 1: Retrieve top-k chunks
from course_utils import lab5_retrieve_chunks

query = "What causes the Northern Lights?"
chunk_size = 200  # You will vary this later
top_k = 3

retrieved = lab5_retrieve_chunks(query, k=top_k, chunk_size=chunk_size)
print("Retrieved chunks:")
for i, r in enumerate(retrieved):
    print(f"[{i+1}] {r[:200]}...")


In [ ]:

# Step 2: Generate an answer based on retrieved context
from course_utils import lab5_generate_answer

# TODO: Modify the code below to pass both the query and the retrieved chunks
# to the generation helper. Observe how context changes answers.

response = lab5_generate_answer(
    query=query,
    retrieved_docs=retrieved  # TODO: confirm correct variable name
)

print("Generated answer:\n", response)



## Part 2 – Run a Chunking & Retrieval Experiment

Now let’s loop over different **chunk sizes** and **top-k** values to measure their effects.

We’ll collect results into a small table and compute **recall@k** and **hallucination rate**.


In [ ]:

from course_utils import lab5_evaluate_recall_at_k, lab5_estimate_hallucination_rate
import pandas as pd

# Example experiment parameters
chunk_sizes = [100, 200, 400]
k_values = [2, 4]

results = []

# TODO: complete the experiment loop
for cs in chunk_sizes:
    for k in k_values:
        retrieved = lab5_retrieve_chunks(query, k=k, chunk_size=cs)
        response = lab5_generate_answer(query, retrieved)
        
        recall = lab5_evaluate_recall_at_k(retrieved, query)
        halluc_rate = lab5_estimate_hallucination_rate(response, retrieved)
        
        results.append({
            "chunk_size": cs,
            "k": k,
            "recall@k": recall,
            "hallucination_rate": halluc_rate
        })

df = pd.DataFrame(results)
df



## Part 3 – Try a Gradio RAG App 🎛️

Use the small demo app below to try different queries and see how retrieval + generation interact.

You can modify the **chunk size** and **top-k** interactively.


In [ ]:

import gradio as gr
from course_utils import lab5_build_demo

# TODO: Customize default chunk size and top-k
demo = lab5_build_demo(default_chunk_size=200, default_k=3)
demo.launch()



## Results

Summarize your main observations:

- How did recall@k change with chunk size?
- How did hallucination rate change?
- Did any trends surprise you?

## Conclusion

Was your hypothesis supported? Why or why not?  
What tradeoffs did you notice between retrieval depth and accuracy?



## Post-Lab Reflection

Answer briefly (2–4 sentences each):

1. What did you learn about how retrieval affects generation?  
2. Describe one surprising behavior you observed in the RAG pipeline.  
3. What’s one next step you’d take to make this system more reliable?

> Edit this cell and type your answers.


In [ ]:

# @title ✅ Run Checks for Lab 5

print("Running Lab 5 checks...")

def _test_metrics():
    import numpy as np
    recalls = [0.5, 0.75, 1.0]
    halluc = [0.1, 0.3, 0.2]
    assert all(0 <= r <= 1 for r in recalls), "recall@k out of range"
    assert all(0 <= h <= 1 for h in halluc), "hallucination rate out of range"
    print("✅ Metric sanity checks passed.")

def _test_demo():
    try:
        from course_utils import lab5_build_demo
        demo = lab5_build_demo()
        print("✅ Demo function found.")
    except Exception as e:
        print("⚠️ Demo import failed:", e)

_test_metrics()
_test_demo()

print("All checks complete.")
